In [23]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from Bio import SeqIO


pep_hits_long = pd.read_csv('long_091826.csv')

In [24]:
lassa = pep_hits_long[pep_hits_long['species'] == 'Mammarenavirus lassaense'].copy()

In [25]:
lassa

,peptide,individual,z_score,rpk,sequence,species,genus,family,protein_std,accession,cohort,category,reactive
94,AHC95546.1|polymerase|Mammarenavirus_lassaense...,C-109-3_4_E7,-0.655047,0.263884,TDDAVVAEDDIEQLIYQFKRASPILRFLYSDIEGEEDRKNVQVVKE,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,polymerase,AHC95546.1,SL,Household contact,False
95,AHC95546.1|polymerase|Mammarenavirus_lassaense...,C-109-3_4_E7,-0.735745,0.900343,RESTSRSITEDFFYSNYQNGVVPSHISSVLDMGQGILHNTSDFYAL,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,polymerase,AHC95546.1,SL,Household contact,False
96,AHC95546.1|polymerase|Mammarenavirus_lassaense...,C-109-3_4_E7,-1.414275,0.000000,LRQKVIYSGAVNLDDDKIPTIVKTIQNKLSSTFTRGAQKLLSEAIN,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,polymerase,AHC95546.1,SL,Household contact,False
108,AIT17175.1|polymerase|Mammarenavirus_lassaense...,C-109-3_4_E7,1.487771,2.520960,RLCESLSMTSGRLSGVESLNVLLDNRSNHYEEVITSCHQGINNKLT,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,polymerase,AIT17175.1,SL,Household contact,False
109,AIT17179.1|polymerase|Mammarenavirus_lassaense...,C-109-3_4_E7,-0.807694,0.180069,SFDVSGVIPTITYQRSEEEKFPYVTGDVELLRTTDLERLSSLSLAL,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,polymerase,AIT17179.1,SL,Household contact,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
615936,WKK95338.1|nucleoprotein|Mammarenavirus_lassae...,HBDB-115,0.083291,0.469034,KAGSNGSNKSLQSAGFTAGLTYSQLMTLKDAMLQLDPNAKTWMDIE,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,nucleoprotein,WKK95338.1,HBDB,US healthy,False
615947,WNI01985.1|nucleoprotein|Mammarenavirus_lassae...,HBDB-115,2.399836,3.497126,KVGTAGSNKSLQSAGFPTGLTYSQLMTLKDSMMQLDPSAKTWIDIE,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,nucleoprotein,WNI01985.1,HBDB,US healthy,False
615971,WPS90786.1|polymerase|Mammarenavirus_lassaense...,HBDB-115,-0.553236,1.024380,DNRSSHYEEIIASCHQGINNKLTAHEVKLQIEEEYQVFRNRLRKGE,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,polymerase,WPS90786.1,HBDB,US healthy,False
615972,WPS90786.1|polymerase|Mammarenavirus_lassaense...,HBDB-115,1.492352,0.639764,VLEAVDDWVDFKGYALCYSKSRKKVMVHSSGGKLRLKGRTCEELVK,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,polymerase,WPS90786.1,HBDB,US healthy,False


## importing the reference sequences fetched from UniProt (UP000002473_11622.fasta)

In [26]:
records = list(SeqIO.parse('UP000002473_11622_LASV_Josiah_reference.fasta', 'fasta'))

for r in records:
    print(r.id, len(r.seq))

sp|O09705|L_LASSJ 2218
sp|O73557|Z_LASSJ 99
sp|P08669|GLYC_LASSJ 491
sp|P13699|NCAP_LASSJ 569


In [27]:
reference_by_protein = {
    "polymerase": str(next(r.seq for r in records if "O09705" in r.id)),
    "glycoprotein": str(next(r.seq for r in records if "P08669" in r.id)),
    "nucleoprotein": str(next(r.seq for r in records if "P13699" in r.id)),
    "z protein": str(next(r.seq for r in records if "O73557" in r.id)),
}

In [28]:
print(lassa['category'].value_counts())
print(lassa['sequence'].nunique())
print(lassa['protein_std'].unique())

category
Household contact    21052
Lassa survivor        9120
US healthy            6612
Name: count, dtype: int64
76
['polymerase' 'glycoprotein' 'nucleoprotein' 'z protein']


## import PairwiseAligner and use BLOSUM62 matrix for local protein alignment

In [29]:
from Bio.Align import PairwiseAligner
from Bio.Align import substitution_matrices

aligner = PairwiseAligner()
aligner.mode = 'local' #want the peptide to find its best location within the reference

aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

## first nucleoprotein

In [30]:
ref = reference_by_protein['nucleoprotein']

np = lassa[lassa['protein_std'].eq("nucleoprotein")].copy()

unique_np_peptides = np['sequence'].drop_duplicates()

## define mapping function

In [31]:
def map_peptide_to_reference(peptide, reference, aligner):
    """
    goal: map a peptide onto a reference protein. 

    output: returns a dictionary containing
    - alignment score
    - reference start/end (1-based)
    - reference positions covered by aligned peptide residues
    """

    #if no exact substring match, use local alignment
    alignment = aligner.align(reference, peptide)[0] #[0] to extract the first and best-scoring Alignment object from the results

    #aligner.align[0].aligned returns the start and end coordinates of the aligned segments for the target and query sequences
    ref_blocks, peptide_blocks = alignment.aligned

    ref_positions = []

    for (ref_start, ref_end), (pep_start, pep_end) in zip(ref_blocks, peptide_blocks):
        
        for i in range(ref_end - ref_start):
            ref_positions.append(ref_start + i + 1) #1-based

    #overall peptide coordinates
    pep_start = peptide_blocks[0][0] + 1
    pep_end = peptide_blocks[-1][1] #take the last block,so -1, in case there are more than one aligned blocks. and [1] for the end

    #total number of peptide residues included in the aligned blocks
    aligned_length = sum(end - start for start, end in peptide_blocks)

    #total number 

    return{
        'score': alignment.score, 
        'ref_start': ref_positions[0],
        'ref_end': ref_positions[-1],
        'ref_positions': ref_positions, 
        'pep_start' : pep_start, 
        'pep_end': pep_end,
        'aligned_length': aligned_length
    }


In [32]:
mapping = []

for peptide in unique_np_peptides:
    result = map_peptide_to_reference(peptide, ref, aligner)

    mapping.append({
        'sequence': peptide, 
        'score': result['score'], 
        'ref_start': result['ref_start'], 
        'ref_end': result['ref_end'], 
        'ref_positions': result['ref_positions']

    })

np_mapping = pd.DataFrame(mapping)

In [33]:
#check the length of peptide alignment, should be ~46

np_mapping['alignment_length'] = np_mapping['ref_positions'].str.len()

np_mapping['alignment_length'].describe()

#inspect the shortest mappings: 
np_mapping.sort_values("alignment_length").head(10)

,sequence,score,ref_start,ref_end,ref_positions,alignment_length
0,TERPLSSGVYMGNLSSQQLDQRRALLNMIGMAGGSQGNQPSRDGVV,198.0,116,161,"[116, 117, 118, 119, 120, 121, 122, 123, 124, ...",46
1,ALLNMIGMAGGSQGNQPSRDGVVRVWDVKNADLLNNQFGTMPSLTL,204.0,139,184,"[139, 140, 141, 142, 143, 144, 145, 146, 147, ...",46
2,KPGNTGSNKSLQSAGFAAGLTYSQLMTLKDSMLQLDPNAKTWIDIE,190.0,346,391,"[346, 347, 348, 349, 350, 351, 352, 353, 354, ...",46
3,KVGTTGSNKSLQSAGFPAGLTYSQLMTLKDSMMQLDPSAKTWIDIE,183.0,346,391,"[346, 347, 348, 349, 350, 351, 352, 353, 354, ...",46
4,KPGNNGSNRSLQSAGFPAGLTYSQLMTLKDSMLQLDPNAKTWMDIE,196.0,346,391,"[346, 347, 348, 349, 350, 351, 352, 353, 354, ...",46
5,KAGSNGSNKSLQSAGFTAGLTYSQLMTLKDAMLQLDPNAKTWMDIE,216.0,346,391,"[346, 347, 348, 349, 350, 351, 352, 353, 354, ...",46
6,KVGTAGSNKSLQSAGFPTGLTYSQLMTLKDSMMQLDPSAKTWIDIE,177.0,346,391,"[346, 347, 348, 349, 350, 351, 352, 353, 354, ...",46
